Esse fluxo é uma tentativa de utilizar o Kafka, a intenção é criar uma instancia que processe as mensagens recebidas por um script Python <br><br>
E depois disso o producer irá operar ao receber as mensagens <br><br>
comecei a fazer a configuração do AWS, mas obtive problemas por questão de bloqueio do AWS Academ. <br><br>
Optei por fazer a tentativa na maquina local com o Docker...<br><br>
 Posteriormente vou reproduzir em uma conta pessoal da AWS para conseguir fazer os testes

## Fluxo da AWS 

In [1]:
import os
import time
import boto3
import dotenv
from botocore.exceptions import ClientError

dotenv.load_dotenv()

True

Carregaremos as variaveis de ambiente e posteriormente faremos a conexão no boto3 com as variaveis carregadas

In [2]:
ID_CONTA = os.getenv("ID_CONTA")
AWS_REGION = os.getenv("AWS_REGION")
AWS_ACCESS_KEY_ID=os.getenv("AWS_ACESS_KEY_ID")
AWS_SECRET_ACCESS_KEY=os.getenv("AWS_SECRET_ACESS_KEY")
#AWS_SESSION_TOKEN=os.getenv("aws_session_token") Token nao é necessario para fora do AWS Academy

In [3]:
# Criando a cessão 
session = boto3.Session(
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    #aws_session_token=AWS_SESSION_TOKEN,
    region_name=AWS_REGION
)

# Inicializamos a conexão no Ec2 
ec2 = session.client('ec2')
print(f"Conectado com sucesso na região: {session.region_name}")


Conectado com sucesso na região: us-east-1


Criando uma VPC 

In [4]:
print("Criando VPC...")
vpc = ec2.create_vpc(
    CidrBlock='10.0.0.0/16',
    TagSpecifications=[
        {
            'ResourceType': 'vpc',
            'Tags': [{'Key': 'Name', 'Value': 'fiap-msk-vpc'}]
        }
    ]
)
vpc_id = vpc['Vpc']['VpcId']
print(f"VPC criada: {vpc_id}")
# Modifica os atributos da VPC para habilitar nomes DNS (necessário para o MSK)
ec2.modify_vpc_attribute(VpcId=vpc_id, EnableDnsHostnames={'Value': True})
ec2.modify_vpc_attribute(VpcId=vpc_id, EnableDnsSupport={'Value': True})

Criando VPC...
VPC criada: vpc-0f91f6cc0d6b482f4


{'ResponseMetadata': {'RequestId': '104669f2-42fd-4d82-a56d-1c9a178e66e9',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '104669f2-42fd-4d82-a56d-1c9a178e66e9',
   'cache-control': 'no-cache, no-store',
   'strict-transport-security': 'max-age=31536000; includeSubDomains',
   'content-type': 'text/xml;charset=UTF-8',
   'content-length': '225',
   'date': 'Sat, 04 Jul 2026 16:38:38 GMT',
   'server': 'AmazonEC2'},
  'RetryAttempts': 0}}

verificando zonas de disponibilidade e subnets 

Zonas de Disponibilidade (AZs): São os Data Centers físicos existentes dentro da região. Cada um desses grupos de data centers é uma Zona de Disponibilidade. 
Para rodar o Kafka optamos por usar duas zonas de disponibilidade diferentes, ou seja dois Brokers. Se uma deixar de funcionar a outra assume. 

já os subnets sao as divisões da VPC (Virtual Private Cloud), Se as zonas de disponibilidade são os datacenters dentro de uma região, os subnets são as salas dentro das zonas de disponibilidade


> Essa parte do código é utilizada para verificar as zonas de disponibilidade, como fiz no academy tive problema para recupera-las
por conta de bloqueio de autorização 
encontrei o nome das zonas disponiveis na região que o academy forneceu e utilizei aqui hardcode mesmo 

In [6]:
#--------
# azs = ec2.describe_availability_zones(
#     Filters=[{'Name': 'state', 'Value': 'available'}]
# )['AvailabilityZones']
az1 = f"{AWS_REGION}a"  # Resultará em 'us-east-1a'
az2 = f"{AWS_REGION}b"  # Resultará em 'us-east-1b'
print(f"Zonas de disponibilidade definidas manualmente: {az1} e {az2}")

Zonas de disponibilidade definidas manualmente: us-east-1a e us-east-1b


Criando os subnet

In [7]:
print(f"Criando Subnet 1 em {az1}...")
subnet1 = ec2.create_subnet(
    VpcId=vpc_id,
    CidrBlock='10.0.1.0/24',
    AvailabilityZone=az1,
    TagSpecifications=[
        {
            'ResourceType': 'subnet',
            'Tags': [{'Key': 'Name', 'Value': 'fiap-msk-subnet-1'}]
        }
    ]
)
subnet1_id = subnet1['Subnet']['SubnetId']

print(f"Criando Subnet 2 em {az2}...")
subnet2 = ec2.create_subnet(
    VpcId=vpc_id,
    CidrBlock='10.0.2.0/24',
    AvailabilityZone=az2,
    TagSpecifications=[
        {
            'ResourceType': 'subnet',
            'Tags': [{'Key': 'Name', 'Value': 'fiap-msk-subnet-2'}]
        }
    ]
)
subnet2_id = subnet2['Subnet']['SubnetId']
print(f"Subnets criadas: {subnet1_id} e {subnet2_id}")


Criando Subnet 1 em us-east-1a...
Criando Subnet 2 em us-east-1b...
Subnets criadas: subnet-0da818ecf9f3db006 e subnet-0f24279ba0a1307b9


Por fim configurar o Internet Gateway, são os nossos acessos as zonas as nossas subnets 

**Route Table** é a definição das nossas rotas de rede para as subnet

In [8]:
# Criando e associando o Internet Gateway
print("Criando Internet Gateway...")
igw = ec2.create_internet_gateway(
    TagSpecifications=[
        {
            'ResourceType': 'internet-gateway',
            'Tags': [{'Key': 'Name', 'Value': 'fiap-msk-igw'}]
        }
    ]
)
igw_id = igw['InternetGateway']['InternetGatewayId']
ec2.attach_internet_gateway(InternetGatewayId=igw_id, VpcId=vpc_id)
print(f"Internet Gateway anexado: {igw_id}")
# Criando uma Route Table
print("Criando Tabela de Roteamento...")
rt = ec2.create_route_table(
    VpcId=vpc_id,
    TagSpecifications=[
        {
            'ResourceType': 'route-table',
            'Tags': [{'Key': 'Name', 'Value': 'fiap-msk-rt'}]
        }
    ]
)
rt_id = rt['RouteTable']['RouteTableId']
# Adicionando a rota de saída padrão (0.0.0.0/0) e apontando para o Internet Gateway (igw_id)
ec2.create_route(
    RouteTableId=rt_id,
    DestinationCidrBlock='0.0.0.0/0',
    GatewayId=igw_id
)
# Por fim associar a Route Table com ambas as subnets
ec2.associate_route_table(SubnetId=subnet1_id, RouteTableId=rt_id)
ec2.associate_route_table(SubnetId=subnet2_id, RouteTableId=rt_id)
print(f"Tabela de Roteamento {rt_id} associada às subnets.")
print("\n✅ Infraestrutura de rede inicial criada com sucesso!")

Criando Internet Gateway...
Internet Gateway anexado: igw-010b13b9e307263de
Criando Tabela de Roteamento...
Tabela de Roteamento rtb-0ee961276fd3b24aa associada às subnets.

✅ Infraestrutura de rede inicial criada com sucesso!


Conectar ao S3 

In [9]:
s3 = session.client('s3')
bucket_name = os.getenv("BUCKET_NAME")
print(f"Nome do bucket a ser criado: {bucket_name}")

Nome do bucket a ser criado: fiap-postech-challenge-002


In [11]:
s3 = session.client('s3')

nome_do_bucket = os.getenv("BUCKET_NAME")

response = s3.list_buckets()
buckets_existentes = [bucket['Name'] for bucket in response['Buckets']]

try:
    # verifica a existencia do bucket da Fiap na AWS conectada
    s3.head_bucket(Bucket=nome_do_bucket)
    print(f"Bucket '{nome_do_bucket}' já existe.")
except ClientError as e:
    # Se der erro 404 (Not Found), significa que o bucket não existe então iremos cria-lo
    error_code = e.response['Error']['Code']
    if error_code == '404':
        s3.create_bucket(Bucket=nome_do_bucket)
        print(f"Bucket '{nome_do_bucket}' criado com sucesso.")
    else:
        # Outro erro
        print(f"Erro ao acessar o bucket: {e}")

# Ativar versionamento no bucket (boa prática e exigido pelo laboratório do Glue)
print("Ativando versionamento no bucket...")
s3.put_bucket_versioning(
    Bucket=bucket_name,
    VersioningConfiguration={'Status': 'Enabled'}
)
print("[INFO] Versionamento ativado com sucesso!")

Bucket 'fiap-postech-challenge-002' criado com sucesso.
Ativando versionamento no bucket...
[INFO] Versionamento ativado com sucesso!


Criando o security Group, para servir como um Firewall de controle de entrada e saida. 
iremos liberar as portas
9092 para o Kafka (PLAINTEXT)  <br><br>
O professor utilizou a porta 2181 Para o Zookeper (gerenciador dos Cluster), mas a partir da versão 3.X do Kafka o Zookeper deixa de ser necessário <br> <br>
optei por tentar esse caminho 

In [13]:
# Nome do Security Group configurado no .env
sg_name = os.getenv("SG_NAME")

print(f"Nome do Security Group a ser criado: {sg_name}")

try:
    # Criando SG na VPC 
    print("Criando o Security Group...")
    sg = ec2.create_security_group(
        GroupName=sg_name,
        Description="SG for MSK Provisioned lab",
        VpcId=vpc_id # Variável contendo o ID da VPC criada anteriormente
    )
    sg_id = sg['GroupId']
    print(f"[SUCESSO] SG criado com ID: {sg_id}")

    # Configurando as portas para acesso
    print("[INFO] Adicionando regras de entrada para o Kafka...")
    ec2.authorize_security_group_ingress(
        GroupId=sg_id,
        IpPermissions=[
            # Porta 9092 para a rede interna da VPC (10.0.0.0/16)
            {
                'IpProtocol': 'tcp',
                'FromPort': 9092,
                'ToPort': 9092,
                'IpRanges': [{'CidrIp': '10.0.0.0/16', 'Description': 'porta para o Kafka PLAINTEXT'}]
            },
            # # Porta 2181 Utilizada para o Zookeper, foi removida 
            # {
            #     'IpProtocol': 'tcp',
            #     'FromPort': 2181,
            #     'ToPort': 2181,
            #     'IpRanges': [{'CidrIp': '10.0.0.0/16', 'Description': 'ZooKeeper'}]
            # },
             # Porta 8090 (Kafka UI) - Permite acessar o gerenciador do Kafka no seu navegador
            {
                'IpProtocol': 'tcp',
                'FromPort': 8090,
                'ToPort': 8090,
                'IpRanges': [{'CidrIp': '0.0.0.0/0', 'Description': 'Acesso ao Kafka UI'}]
            },
            # Porta 22 (SSH) - Permite que você conecte no terminal da EC2 pelo computador
            {
                'IpProtocol': 'tcp',
                'FromPort': 22,
                'ToPort': 22,
                'IpRanges': [{'CidrIp': '0.0.0.0/0', 'Description': 'Acesso via SSH'}]
            },
            # Permissão de tráfego interno completa entre os recursos que usam esse mesmo Security Group
            
            {
                'IpProtocol': 'tcp',
                'FromPort': 0,
                'ToPort': 65535,
                'UserIdGroupPairs': [{'GroupId': sg_id, 'Description': 'Porta para trafego interno do Glue/Kafka'}]
            }
        ]
    )
    print("[SUCESSO] Regras de firewall configuradas com sucesso!")

except ec2.exceptions.ClientError as e:
    # Tratamento caso o Security Group já tenha sido criado anteriormente
    if 'InvalidGroup.Duplicate' in str(e):
        print(f"[INFO] O Security Group '{sg_name}' já existe. Recuperando ID...")
        existing_sgs = ec2.describe_security_groups(
            Filters=[
                {'Name': 'group-name', 'Values': [sg_name]},
                {'Name': 'vpc-id', 'Values': [vpc_id]}
            ]
        )
        sg_id = existing_sgs['SecurityGroups'][0]['GroupId']
        print(f"[SUCESSO] ID do Security Group existente recuperado: {sg_id}")
    else:
        print(f"[ERRO] : {e}")

# Mantém o ID salvo na memória para o próximo passo
print(f"\nSG_ID ativo: {sg_id}")


Nome do Security Group a ser criado: fiap-msk-provisioned-sg
Criando o Security Group...
[INFO] O Security Group 'fiap-msk-provisioned-sg' já existe. Recuperando ID...
[SUCESSO] ID do Security Group existente recuperado: sg-0c8fa8275200c9fd5

SG_ID ativo: sg-0c8fa8275200c9fd5


In [ ]:
# #Deletar o SG 
# ec2.delete_security_group(GroupId=sg_id)
# print("SG Deletado.")

Inicializando o Kafka Via MSK 
Não optei por utilizar o MSK quero criar o meu próprio Kafka por via EC2 por isso nao utilizei o que o professor fez

In [ ]:
# # Inicializar o cliente do Kafka (MSK)
# kafka = session.client('kafka')

# cluster_name = os.getenv("CLUSTER_NAME")
# print(f"Iniciando a criação do cluster MSK (KRaft): {cluster_name}")

# try: 
#     response = kafka.create_cluster(
#         ClusterName=cluster_name,
#         KafkaVersion="3.7.x.kraft",  # kraft ativa o modo KRaft (Sem ZooKeeper)
#         NumberOfBrokerNodes=2,       # broker em cada subnet
#         BrokerNodeGroupInfo={
#             'InstanceType': 'kafka.t3.small',
#             'ClientSubnets': [subnet1_id, subnet2_id], # Criadas no Passo 0
#             'SecurityGroups': [sg_id],                 # Criado no Passo 2
#             'StorageInfo': {
#                 'EbsStorageInfo': {
#                     'VolumeSize': 10  # 10 GB de armazenamento
#                 }
#             }
#         },
#         EncryptionInfo={
#             'EncryptionInTransit': {
#                 'ClientBroker': 'PLAINTEXT', # Tráfego interno de dados sem criptografia
#                 'InCluster': False
#             }
#         }
#     )
#     cluster_arn = response['ClusterArn']
#     print(f"[SUCESSO] Cluster MSK enviado para criação com sucesso!")
#     print(f"Cluster ARN: {cluster_arn}")
#     print("\n[INFO] O cluster leva cerca de 15 a 20 minutos para ficar 'ACTIVE'.")

# except kafka.exceptions.BadRequestException as e:
#     if 'already exists' in str(e):
#         print(f"[INFO] O cluster '{cluster_name}' já existe. Recuperando detalhes...")
#         clusters = kafka.list_clusters(ClusterNameFilter=cluster_name)['ClusterInfoList']
#         cluster_arn = clusters[0]['ClusterArn']
#         print(f"[SUCESSO] ARN do cluster recuperado: {cluster_arn}")
#     else:
#         print(f"[ERRO] falha ao criar o cluster: {e}")
# except Exception as e:
#     print(f"[ERRO]: {e}")


Esse fluxo até aqui foi feito pelo professor na aula gravada [LINK](https://zoom.us/rec/play/iLSLbM9bB2XKhY9FEyhrviWsoJ1AlJAIcEQ7Xk9dJzpkQz6CZprnlFd5lkmHJ1M9FOva-P_yd_zmRAga.2hhqpObycOC0ZS92?accessLevel=meeting&canPlayFromShare=true&from=share_recording_detail&startTime=1781647585000&oldStyle=true&componentName=rec-play&originRequestUrl=https%3A%2F%2Fzoom.us%2Frec%2Fshare%2FB8RCJOVlnPMmobZvMgXKdPyePQeA7Kj6R_fXDOxGS-TebFC36ngCgCELqnAG0-Jv.dOgH2tzEroMkVNZ4%3FstartTime%3D1781647585000)

Utilizei como base para gerar todo o código e adaptar para script em python em vez do script padrão em CLI do AWS, assim os demais colegas pudessem utilizar o fluxo em suas maquinas sem grande dificuldade

**ATENÇÃO AO LIGAR O CLUSTER ELE PROVISIONARÁ UM CUSTO DE U$ 0,07 POR HORA PARA CADA BROKER, <br> ENTÃO É IMPORTANTE DESLIGAR O CLUSTER AO FINAL DOS TESTES**


Por conta do custo optei por tentar fazer algo diferente e subir um Kafka em uma EC2 para aprender um pouco mais sofre o funcionamento da ferramenta.
aproveitei a oportunidade para fazer o estudo do terraform que é utilizado para automatizar a criação de engenharia em cloud

O script a seguir gera o Ec2 e está comentado para não gerar novamente sem querer

In [ ]:
# # Script que a EC2 executará ao ligar: instala Docker e roda o Kafka 
# # o ts3.small é uma instancia de 2gb de ram da ec2 aws e atendera a necessidade do nosso kafka
# #O inicio do script /bin/bash sao comandos Linux, responsaveis por atualizar o sistema, instalar o docker e criar as pastas do projeto 
# #o services está reconstruindo o nosso docker-compose-yml em nosso Ec2  
# #O IP 169.254.169.254 é o IP Privado e Interno da AWS utilizado para que a maquina consiga saber o seu verdadeiro IP Dinamico e assim gerar o Ec2 


# user_data_script = """#!/bin/bash
# sudo apt-get update -y
# sudo apt-get install -y docker.io docker-compose
# sudo systemctl enable docker
# sudo systemctl start docker

# mkdir -p /home/ubuntu/kafka
# cat <<'INNER_EOF' > /home/ubuntu/kafka/docker-compose.yml
# version: '3.8'
# services:
#   kafka:
#     image: apache/kafka:3.7.0
#     container_name: kafka-fiap
#     ports:
#       - "9092:9092"
#     environment:
#       KAFKA_NODE_ID: 1
#       KAFKA_PROCESS_ROLES: broker,controller
#       KAFKA_LISTENERS: PLAINTEXT://0.0.0.0:9092,CONTROLLER://0.0.0.0:9093,INTERNAL://0.0.0.0:9094
#       KAFKA_ADVERTISED_LISTENERS: PLAINTEXT://PUBLIC_IP_PLACEHOLDER:9092,INTERNAL://kafka:9094
#       KAFKA_CONTROLLER_LISTENER_NAMES: CONTROLLER
#       KAFKA_LISTENER_SECURITY_PROTOCOL_MAP: CONTROLLER:PLAINTEXT,PLAINTEXT:PLAINTEXT,INTERNAL:PLAINTEXT
#       KAFKA_INTER_BROKER_LISTENER_NAME: INTERNAL
#       KAFKA_CONTROLLER_QUORUM_VOTERS: 1@localhost:9093
#       KAFKA_OFFSETS_TOPIC_REPLICATION_FACTOR: 1
#       KAFKA_TRANSACTION_STATE_LOG_REPLICATION_FACTOR: 1
#       KAFKA_TRANSACTION_STATE_LOG_MIN_ISR: 1
#       KAFKA_GROUP_INITIAL_REBALANCE_DELAY_MS: 0
#       KAFKA_NUM_PARTITIONS: 3
#       KAFKA_AUTO_CREATE_TOPICS_ENABLE: "true"
#       CLUSTER_ID: MKU30EVBNTcwNTJENDM2Qk
#     healthcheck:
#       test: [ "CMD-SHELL", "nc -z localhost 9092 || exit 1" ]
#       interval: 10s
#       timeout: 10s
#       retries: 5
#       start_period: 30s

#   kafka-ui:
#     image: provectuslabs/kafka-ui:latest
#     container_name: kafka-fiap-ui
#     depends_on:
#       kafka:
#         condition: service_healthy
#     ports:
#       - "8090:8080"
#     environment:
#       KAFKA_CLUSTERS_0_NAME: checkout-cluster
#       KAFKA_CLUSTERS_0_BOOTSTRAPSERVERS: kafka:9094
#       DYNAMIC_CONFIG_ENABLED: "true"
# INNER_EOF

# # Aguarda a rede subir e busca o IP público do EC2 via serviço de metadados da AWS (IMDSv2)
# sleep 15
# TOKEN=$(curl -s -X PUT "http://169.254.169.254/latest/api/token" -H "X-aws-ec2-metadata-token-ttl-seconds: 21600")
# PUBLIC_IP=$(curl -s -H "X-aws-ec2-metadata-token: $TOKEN" http://169.254.169.254/latest/meta-data/public-ipv4)

# # Substitui o placeholder com o IP real da EC2 no docker-compose.yml
# sed -i "s/PUBLIC_IP_PLACEHOLDER/$PUBLIC_IP/g" /home/ubuntu/kafka/docker-compose.yml

# # Executa o docker-compose
# cd /home/ubuntu/kafka
# sudo docker-compose up -d
# """

# # Executa o comando de criação da EC2
# print("Criando instância EC2...")
# res = ec2.run_instances(
#     ImageId='ami-0e2c8caa4b6378d8c', # Imagem Ubuntu 24.04 LTS padrão da região us-east-1
#     InstanceType='t3.small',
#     MinCount=1,
#     MaxCount=1,
#     KeyName='chave-fiap-kafka', #essa chave precisa ser criada na AWS, em EC2 -> Rede e segurança
#     NetworkInterfaces=[
#         {
#             'SubnetId': subnet1_id, # Usando a Subnet criado anteriormente
#             'DeviceIndex': 0,
#             'AssociatePublicIpAddress': True,
#             'Groups': [sg_id] # Usando o Security Group criado anteriormente
#         }
#     ],
#     UserData=user_data_script,
#     TagSpecifications=[
#         {
#             'ResourceType': 'instance',
#             'Tags': [{'Key': 'Name', 'Value': 'fiap-kafka-ec2'}]
#         }
#     ]
# )

# inst_id = res['Instances'][0]['InstanceId']
# print(f"\n[SUCESSO] Instância EC2 iniciada com sucesso! ID: {inst_id}")


Criando instância EC2...

[SUCESSO] Instância EC2 iniciada com sucesso! ID: i-03ec4cb054019acbc



![Bucket Criado no S3](../reports/figures/EC2_criado.png)


Desligar e ligar a maquina

In [ ]:
# Célula para DESLIGAR a máquina
def desligar_ec2():
    res = ec2.describe_instances(Filters=[{'Name': 'tag:Name', 'Values': ['fiap-kafka-ec2']}])
    if res['Reservations']:
        inst_id = res['Reservations'][0]['Instances'][0]['InstanceId']
        ec2.stop_instances(InstanceIds=[inst_id])
        print(f"Desligando a máquina EC2 ({inst_id})...")
    else:
        print("Nenhuma máquina encontrada.")

# Célula para LIGAR a máquina
def ligar_ec2():
    res = ec2.describe_instances(Filters=[{'Name': 'tag:Name', 'Values': ['fiap-kafka-ec2']}])
    if res['Reservations']:
        inst_id = res['Reservations'][0]['Instances'][0]['InstanceId']
        ec2.start_instances(InstanceIds=[inst_id])
        print(f"Iniciando a máquina EC2 ({inst_id})...")
    else:
        print("Nenhuma máquina encontrada.")

